In [2]:
from lxml import etree

In [1]:
import re

with open('data/0003-001/0003-001.xml', 'r', encoding='utf-8') as f:
    content = f.read()

In [3]:
def fix_malformed_xml(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Fix angle brackets in form_original attributes (any position)
    content = re.sub(
        r'form_original="([^"]*?)<([^"]*?)"',
        lambda m: f'form_original="{m.group(1)}&lt;{m.group(2)}"',
        content
    )
    content = re.sub(
        r'form_original="([^"]*?)>([^"]*?)"',
        lambda m: f'form_original="{m.group(1)}&gt;{m.group(2)}"',
        content
    )
    
    # Fix unescaped quotes in form and lemma attributes
    content = re.sub(
        r'(form|lemma)="""',
        r'\1="&quot;"',
        content
    )
    
    # Fix malformed entities by escaping the ampersand
    content = re.sub(
        r'&(?!lt;|gt;|amp;|quot;|apos;|#\d+;|#x[0-9a-fA-F]+;)',
        r'&amp;',
        content
    )
    
    return etree.fromstring(content.encode('utf-8'))

In [4]:
content = fix_malformed_xml('data/0003-001/0003-001.xml')

In [6]:
xpath = f"//sentence[@id='{11}']/word"
glaux_elements = content.xpath(xpath)

In [10]:
[e.get('form') for e in glaux_elements]

['δηλοῖ',
 'δέ',
 'μοι',
 'καὶ',
 'τόδε',
 'τῶν',
 'παλαιῶν',
 'ἀσθένειαν',
 'οὐχ',
 'ἥκιστα',
 '·']

In [5]:
root = etree.fromstring(content.encode('utf-8'))

AttributeError: 'lxml.etree._Element' object has no attribute 'encode'

In [19]:
import pandas as pd

glosses_lookup = pd.read_csv("data/0085-007/glosses.csv")

In [23]:
glosses_lookup

,Unnamed: 0,greek_id,greek_word,gloss,sent_id
0,0,102401876,μὲν,"but (contrastive particle, marking a contrasti...",0
1,1,102401877,μὲν,"indeed, but (particle used to mark contrast or...",0
2,2,102401878,εὐχῇ,prayer,0
3,3,102401879,τῇδε,this,0
4,4,102401880,πρεσβεύω,I supplicate,0
...,...,...,...,...,...
277,277,102402150,μὲν,"but (contrastive particle, introducing a contr...",19
278,278,102402151,οὐ̓͂ν,"then, indeed (emphatic particle used to introd...",19
279,279,102402152,.,period,19
280,280,1000014928,E,be,19


In [30]:
xpath = f"//sentence[@id='{1}']/word"
glaux_elements = root.xpath(xpath)
print(len(glaux_elements))
def render_span(elem):
    html_template = '<span class="word" data-id="{word_id}" form="{form}" lemma="{lemma}" postag="{postag}" head="{head}" relation="{relation}" gloss="{gloss}">{text}</span>'
    word_id = elem.get("id", "")
    form = elem.get("form", "")
    lemma = elem.get("lemma", "")
    postag = elem.get("postag", "")
    head = elem.get("head", "")
    relation = elem.get("relation", "")

    glosses = glosses_lookup[glosses_lookup['greek_id'] == int(word_id)]
    gloss = glosses['gloss'].values[0] if not glosses.empty else ""
    text = form if form.strip() else ""
    return html_template.format(word_id=word_id, form=form, lemma=lemma, postag=postag, head=head, relation=relation, gloss=gloss, text=text)

html_parts = []
for elem in glaux_elements:
    word_text = elem.get("form", "") or ""
    if word_text.strip():
        span_html = render_span(elem)
        html_parts.append(span_html)

10


In [31]:
from IPython.display import display, HTML
display(HTML("\n".join(html_parts)))

In [33]:
translations = pd.read_csv("data/0085-007/translations.csv")
translations[translations['sent_id'] == 1]

,Unnamed: 0,translation,notes,sent_id
1,1,"From the shrine of Themis, indeed, the second ...","This sentence is from Aeschylus' *Eumenides*, ...",1


In [32]:
html_parts

['<span class="word" data-id="102401876" form="πρῶτον" lemma="πρότερος" postag="a-s---nac" head="102401880" relation="ADV" gloss="but (contrastive particle, marking a contrastive clause or discourse shift)">πρῶτον</span>',
 '<span class="word" data-id="102401877" form="μὲν" lemma="μέν" postag="g--------" head="102401880" relation="AuxY" gloss="indeed, but (particle used to mark contrast or continuation in discourse)">μὲν</span>',
 '<span class="word" data-id="102401878" form="εὐχῇ" lemma="εὐχή" postag="n-s---fd-" head="102401880" relation="OBJ" gloss="prayer">εὐχῇ</span>',
 '<span class="word" data-id="102401879" form="τῇδε" lemma="ὅδε" postag="p-s---fd-" head="102401878" relation="ATR" gloss="this">τῇδε</span>',
 '<span class="word" data-id="102401880" form="πρεσβεύω" lemma="πρεσβεύω" postag="v1spia---" head="0" relation="PRED" gloss="I supplicate">πρεσβεύω</span>',
 '<span class="word" data-id="102401881" form="θεῶν" lemma="θεός" postag="n-p---mg-" head="1024

In [35]:
translation_lookup = pd.read_csv("data/0085-007/eumenides_first20_translations.csv")

In [45]:
translation_data = translation_lookup[translation_lookup.id.isin(range(5,8))].itertuples()
for row in translation_data:
    print(f"{row.translation}\n\t{row.notes}")

The name of Phoebus is the appellation.
	The sentence reflects a theological or semantic reflection on the nature of divine names in Greek thought. 'Phoebus' (Φοίβης) is an epithet of Apollo, emphasizing his radiant and purifying attributes. The term παρώνυμον ('appellation') refers to a name given by another or a title bestowed upon someone; here it underscores that the name 'Phoebus' is not merely descriptive but carries authority and identity. This kind of linguistic self-awareness—where a deity's name is explicitly named as an appellation—is characteristic of Aeschylean drama, which often explores themes of identity, legitimacy, and divine speech.
Having left the lake of Delos and the rock, having called upon the shores of the naval warriors of Pallas, he came to this land, the seat of Parnaeus.
	The sentence describes a divine or heroic journey, likely referring to the arrival of a deity (possibly Athena, given her association with Pallas and Delos) in Greece. The lake of Delos 

In [3]:
import pandas as pd
annotations = pd.read_csv("data/0085-007/annotation_lookup.csv")
annotations.head()

,Unnamed: 0,greek_id,form,english_word,gloss,id,author,work
0,0,102401876,πρῶτον,First,first,1,Aeschylus,Eumenides
1,1,102401877,μὲν,of all,indeed,1,Aeschylus,Eumenides
2,2,102401878,εὐχῇ,I pray,prayer,1,Aeschylus,Eumenides
3,3,102401879,τῇδε,to these,these,1,Aeschylus,Eumenides
4,4,102401880,πρεσβεύω,gods,I pray,1,Aeschylus,Eumenides


In [8]:
annotations[18:]['greek_id'] = annotations[18:]['greek_id'] + 1

/var/folders/l_/b5wczsf55z9dmf_dzr3vw6ph0000gr/T/ipykernel_47534/2556750422.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  annotations[18:]['greek_id'] = annotations[18:]['greek_id'] + 1
/var/folders/l_/b5wczsf55z9dmf_dzr3vw6ph0000gr/T/

In [9]:
annotations.to_csv("data/0085-007/annotation_lookup.csv", index=False)

In [1]:
import pandas as pd
from lxml import etree
import re

annotation_df = pd.read_csv("data/0085-007/annotation_lookup.csv")

with open('data/0085-007/0085-007.xml', 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(
    r'form_original="<([^"]*?)>"',
    r'form_original="&lt;\1&gt;"',
    content
)
tree = etree.fromstring(content.encode('utf-8'))

In [2]:
words = tree.xpath('//word')
words[:5]

[<Element word at 0x1291751c0>,
 <Element word at 0x129174040>,
 <Element word at 0x129177f00>,
 <Element word at 0x129177cc0>,
 <Element word at 0x129174a40>]

In [14]:
annotations = pd.read_csv("data/0085-007/eumenides_first20_annotations.csv")

for word in words[:50]:
    word_id = int(word.get('id', 0))
    form = word.get('form', '')
    lemma = word.get('lemma', '')

    print(f"Word ID: {word_id}")
    annotation_row = annotations[annotations['greek_id'] == word_id]
    if not annotation_row.empty:
        print(annotation_row.gloss.iloc[0], f"(form: {form}, lemma: {lemma})")
    else:
        print(f"No annotation found for word ID {word_id} (form: {form}, lemma: {lemma})")
    

Word ID: 102401876
first (form: πρῶτον, lemma: πρότερος)
Word ID: 102401877
indeed (form: μὲν, lemma: μέν)
Word ID: 102401878
prayer (form: εὐχῇ, lemma: εὐχή)
Word ID: 102401879
these (form: τῇδε, lemma: ὅδε)
Word ID: 102401880
I pray (form: πρεσβεύω, lemma: πρεσβεύω)
Word ID: 102401881
of the gods (form: θεῶν, lemma: θεός)
Word ID: 102401882
the (form: τὴν, lemma: ὁ)
Word ID: 102401883
primal prophetess (form: πρωτόμαντιν, lemma: πρωτόμαντις)
Word ID: 102401884
Gaia (form: Γαῖαν, lemma: γῆ)
Word ID: 102401885
No annotation found for word ID 102401885 (form: ·, lemma: ·)
Word ID: 102401886
from (form: ἐκ, lemma: ἐκ)
Word ID: 102401887
indeed (form: δὲ, lemma: δέ)
Word ID: 102401888
of the (form: τῆς, lemma: ὁ)
Word ID: 102401889
Themis (form: Θέμιν, lemma: θέμις)
Word ID: 102401890
No annotation found for word ID 102401890 (form: ,, lemma: ,)
Word ID: 102401891
, (form: ἣ, lemma: ὅς)
Word ID: 102401892
the (form: δὴ, lemma: δή)
Word ID: 102401893
in

In [ ]:
all_tokens = []
for word in tree.xpath('//word'):
    word_id = int(word.get('id', 0))
    form = word.get('form', '')
    lemma = word.get('lemma', '')
    
    all_tokens.append({
        'greek_id': word_id,
        'form': form,
        'gloss': '',  # Empty gloss for missing tokens
        'english_word': '',  # Empty english_word for missing tokens
    })

all_tokens_df = pd.DataFrame(all_tokens)

filled_df = all_tokens_df.merge(
    annotation_df[['greek_id', 'gloss', 'english_word', 'id']], 
    on='greek_id', 
    how='left',
    suffixes=('', '_annotated')
)

filled_df['gloss'] = filled_df['gloss_annotated'].fillna('')
filled_df['english_word'] = filled_df['english_word_annotated'].fillna('')
filled_df # = filled_df[['greek_id', 'form', 'gloss', 'english_word']]

In [14]:
new_annotations = filled_df[filled_df['id'].notna()]

In [15]:
new_annotations.to_csv("data/0085-007/annotation_lookup.csv", index=False)